In [1]:
pip install wordcloud seaborn scipy thefuzz python-Levenshtein


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re

In [3]:
def get_written_dates_and_any_text_file_locations_to_dataframe(root_folder):
    print(f"--- Starting Scan for metadata.json and any associated non-JSON text files ---")
    print(f"Scanning root folder: {os.path.abspath(root_folder)}\n")

    if not os.path.exists(root_folder):
        print(f"Error: The root folder '{root_folder}' does not exist at '{os.path.abspath(root_folder)}'.")
        return pd.DataFrame()

    data_for_df = []
    
    for dirpath, dirnames, filenames in os.walk(root_folder):
        metadata_filepath_in_dir = None
        text_file_path_in_dir = None
        
        non_json_files = []

        # First, identify the metadata.json and any non-JSON files in the current directory
        for filename in filenames:
            if filename.lower() == "metadata.json":
                metadata_filepath_in_dir = os.path.join(dirpath, filename)
            elif filename.lower().endswith(".txt"): # Consider anything that's not JSON as a potential text file
                non_json_files.append(os.path.join(dirpath, filename))

        # Check conditions for a valid pair
        if metadata_filepath_in_dir:
            if len(non_json_files) == 1:
                # Exactly one non-JSON file found, assume it's the text file
                text_file_path_in_dir = non_json_files[0]
                
                print(f"Processing directory: {os.path.abspath(dirpath)}")
                print(f"  Found metadata.json: {metadata_filepath_in_dir}")
                print(f"  Found associated text file: {text_file_path_in_dir}")

                extracted_written_date = None
                try:
                    with open(metadata_filepath_in_dir, 'r', encoding='utf-8') as f:
                        metadata = json.load(f)
                        if 'written_date' in metadata:
                            extracted_written_date = metadata['written_date']
                            print(f"  Extracted 'written_date': '{extracted_written_date}'")
                        else:
                            print(f"  Warning: 'written_date' key NOT found in {metadata_filepath_in_dir}. Setting to None for this entry.")
                except json.JSONDecodeError as e:
                    print(f"  Error: Could not parse JSON from {metadata_filepath_in_dir}: {e}. Setting 'written_date' to None.")
                except Exception as e:
                    print(f"  An unexpected error occurred reading {metadata_filepath_in_dir}: {e}. Setting 'written_date' to None.")
                
                # Add this pair to our DataFrame data
                data_for_df.append({
                    'title': os.path.basename(dirpath),
                    'written_date': extracted_written_date,
                    'text_file_location': os.path.abspath(text_file_path_in_dir)
                })
                
            elif len(non_json_files) == 0:
                print(f"  Note: Found metadata.json in {os.path.abspath(dirpath)} but NO non-JSON text file found. Skipping this directory.")
            else: # len(non_json_files) > 1
                print(f"  Warning: Found metadata.json in {os.path.abspath(dirpath)} but MULTIPLE non-JSON files ({non_json_files}). Cannot determine which is the text file. Skipping this directory.")
        # No else for when metadata.json is not found at all, to keep output cleaner

    print(f"\n--- Scan Complete ---")
    print(f"Total valid pairs (metadata.json + single non-JSON text file): {len(data_for_df)}")
    
    df = pd.DataFrame(data_for_df)

    if df.empty:
        print("No valid pairs of metadata.json and single non-JSON text file found to create a DataFrame.")
    else:
        print("\n--- Resulting DataFrame Head ---")
        print(df.head())
        print(f"\nDataFrame has {len(df)} rows and {len(df.columns)} columns.")
    
    return df

In [4]:
root_directory_to_scan = "data/OCR_Final"

list_of_file_locations = get_written_dates_and_any_text_file_locations_to_dataframe(root_directory_to_scan)

--- Starting Scan for metadata.json and any associated non-JSON text files ---
Scanning root folder: /Users/nevidujayatilleke/Documents/MSC - Research/OCR of Corpus/SiDiaC-v.2.0/data/OCR_Final

Processing directory: /Users/nevidujayatilleke/Documents/MSC - Research/OCR of Corpus/SiDiaC-v.2.0/data/OCR_Final/ධර්ම ප්‍රදීපිකාව හෙවත් මහාබෝධිවංශ පරිකථාව
  Found metadata.json: data/OCR_Final/ධර්ම ප්‍රදීපිකාව හෙවත් මහාබෝධිවංශ පරිකථාව/metadata.json
  Found associated text file: data/OCR_Final/ධර්ම ප්‍රදීපිකාව හෙවත් මහාබෝධිවංශ පරිකථාව/ධර්ම ප්‍රදීපිකාව හෙවත් මහාබෝධිවංශ පරිකථාව.txt
  Extracted 'written_date': '1187 - 1225'
Processing directory: /Users/nevidujayatilleke/Documents/MSC - Research/OCR of Corpus/SiDiaC-v.2.0/data/OCR_Final/දුක්ඛ සත්‍ය වර්ණනා
  Found metadata.json: data/OCR_Final/දුක්ඛ සත්‍ය වර්ණනා/metadata.json
  Found associated text file: data/OCR_Final/දුක්ඛ සත්‍ය වර්ණනා/දුක්ඛ සත්‍ය වර්ණනා.txt
  Extracted 'written_date': ''
Processing directory: /Users/nevidujayatilleke/Documents/MS

In [5]:
list_of_file_locations

,title,written_date,text_file_location
0,ධර්ම ප්‍රදීපිකාව හෙවත් මහාබෝධිවංශ පරිකථාව,1187 - 1225,/Users/nevidujayatilleke/Documents/MSC - Resea...
1,දුක්ඛ සත්‍ය වර්ණනා,,/Users/nevidujayatilleke/Documents/MSC - Resea...
2,ටිබැට් රටේ බුදුදහම,1832 - 1907,/Users/nevidujayatilleke/Documents/MSC - Resea...
3,නිදහසේ මන්ත්‍රය,1901 - 1938,/Users/nevidujayatilleke/Documents/MSC - Resea...
4,පැරණි ගම,1944,/Users/nevidujayatilleke/Documents/MSC - Resea...
...,...,...,...
180,ලියනෝරා නාඩගම,1852 - 1927,/Users/nevidujayatilleke/Documents/MSC - Resea...
181,පානදුරා වාදය,1873,/Users/nevidujayatilleke/Documents/MSC - Resea...
182,බැමිණිටියා සාය හෙවත් කෑරගල පුරාවෘත්තය,,/Users/nevidujayatilleke/Documents/MSC - Resea...
183,කාව්‍යසංග්‍රහව හෙවත් කව් සඟරාව,,/Users/nevidujayatilleke/Documents/MSC - Resea...


In [6]:
list_of_file_locations['text_file_location'][1]

'/Users/nevidujayatilleke/Documents/MSC - Research/OCR of Corpus/SiDiaC-v.2.0/data/OCR_Final/දුක්ඛ සත්\u200dය වර්ණනා/දුක්ඛ සත්\u200dය වර්ණනා.txt'

In [7]:
# Replace empty strings and None with np.nan in 'written_date'
list_of_file_locations['written_date'].replace(['', None], np.nan, inplace=True)

/var/folders/p0/y_n4880x7cjbhx11tzvvqpzh0000gn/T/ipykernel_85850/99294019.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  list_of_file_locations['written_date'].replace(['', None], np.nan, inplace=True)


In [8]:
list_of_file_locations.head()

,title,written_date,text_file_location
0,ධර්ම ප්‍රදීපිකාව හෙවත් මහාබෝධිවංශ පරිකථාව,1187 - 1225,/Users/nevidujayatilleke/Documents/MSC - Resea...
1,දුක්ඛ සත්‍ය වර්ණනා,NaN,/Users/nevidujayatilleke/Documents/MSC - Resea...
2,ටිබැට් රටේ බුදුදහම,1832 - 1907,/Users/nevidujayatilleke/Documents/MSC - Resea...
3,නිදහසේ මන්ත්‍රය,1901 - 1938,/Users/nevidujayatilleke/Documents/MSC - Resea...
4,පැරණි ගම,1944,/Users/nevidujayatilleke/Documents/MSC - Resea...


In [9]:
list_of_file_locations

,title,written_date,text_file_location
0,ධර්ම ප්‍රදීපිකාව හෙවත් මහාබෝධිවංශ පරිකථාව,1187 - 1225,/Users/nevidujayatilleke/Documents/MSC - Resea...
1,දුක්ඛ සත්‍ය වර්ණනා,NaN,/Users/nevidujayatilleke/Documents/MSC - Resea...
2,ටිබැට් රටේ බුදුදහම,1832 - 1907,/Users/nevidujayatilleke/Documents/MSC - Resea...
3,නිදහසේ මන්ත්‍රය,1901 - 1938,/Users/nevidujayatilleke/Documents/MSC - Resea...
4,පැරණි ගම,1944,/Users/nevidujayatilleke/Documents/MSC - Resea...
...,...,...,...
180,ලියනෝරා නාඩගම,1852 - 1927,/Users/nevidujayatilleke/Documents/MSC - Resea...
181,පානදුරා වාදය,1873,/Users/nevidujayatilleke/Documents/MSC - Resea...
182,බැමිණිටියා සාය හෙවත් කෑරගල පුරාවෘත්තය,NaN,/Users/nevidujayatilleke/Documents/MSC - Resea...
183,කාව්‍යසංග්‍රහව හෙවත් කව් සඟරාව,NaN,/Users/nevidujayatilleke/Documents/MSC - Resea...


In [10]:
def clean_words(data, path = True):
    try:
        if path:
            with open(data, 'r', encoding='utf-8') as file:
                text = file.read()
        else:
            text = data
        # Pattern: Split by whitespace OR find and keep the <psi> token
        tokens = re.split(r'(\s+|<psi>)', text)
        
        # Filter out empty strings and pure whitespace left by re.split()
        tokens = [t.strip() for t in tokens if t.strip()]  # FIX: keep '<psi>' so the merge loop below can see it

        # 2. Handle the <psi> token by concatenating adjacent words
        processed_tokens = []
        i = 0
        while i < len(tokens):
            if tokens[i] == '<psi>':
                # Check for preceding and succeeding words
                if i > 0 and i < len(tokens) - 1:
                    # Concatenate the left word, right word, and replace the <psi>
                    new_word = tokens[i-1] + tokens[i+1]
                    # Replace the previous token (which was added in the previous loop iteration)
                    processed_tokens[-1] = new_word
                    i += 2  # Skip <psi> and the concatenated right word
                elif i > 0:
                    # <psi> is at the end, just remove it.
                    i += 1
                elif i < len(tokens) - 1:
                    # <psi> is at the start, just remove it.
                    i += 1
                else:
                    # <psi> is the only token, just remove it.
                    i += 1
            else:
                processed_tokens.append(tokens[i])
                i += 1

        # Re-join the processed tokens into a single string for cleaning
        processed_text = ' '.join(processed_tokens)

        # 3. Remove non-Sinhala characters (keeping Sinhala Unicode range and whitespace)
        cleaned = re.sub(r'[^\u0D80-\u0DFF\s]', '', processed_text)

        # 4. Tokenize by whitespace and count
        words = cleaned.split()
        return words

    except Exception as e:
        print(f"Error reading {path}: {e}")
        return 0

In [11]:
# Word count before removing records without written dates
list_of_file_locations['word_count_bf'] = list_of_file_locations['text_file_location'].apply(lambda x: len(clean_words(x)))

In [12]:
list_of_file_locations['word_count_bf'].head()

0     820
1     967
2    1052
3    1076
4    1627
Name: word_count_bf, dtype: int64

In [13]:
list_of_file_locations['word_count_bf'].sum()

np.int64(229098)

In [14]:
def count_eos_tokens(file_path, token='<eos>'):
    try:
        # Open the file in read mode ('r') with UTF-8 encoding
        with open(file_path, 'r', encoding='utf-8') as f:
            # Read the entire content of the file into a single string
            file_content = f.read()
            
            # Count the occurrences of the token string
            token_count = file_content.count(token)
            
            return token_count
    except FileNotFoundError:
        print(f"Error: The file was not found at path: {file_path}")
        return 0
    except Exception as e:
        print(f"An error occurred: {e}")
        return 0

In [15]:
# Word count before removing records without written dates
list_of_file_locations['sent_count_bf'] = list_of_file_locations['text_file_location'].apply(count_eos_tokens)

In [16]:
list_of_file_locations['sent_count_bf'].head()

0     41
1     48
2     53
3    182
4     92
Name: sent_count_bf, dtype: int64

In [17]:
list_of_file_locations['sent_count_bf'].describe()

count    185.000000
mean      73.097297
std       61.890956
min        1.000000
25%       31.000000
50%       52.000000
75%       99.000000
max      343.000000
Name: sent_count_bf, dtype: float64

In [18]:
list_of_file_locations['sent_count_bf'].sum()

np.int64(13523)

In [19]:
all_text = ""
for path in list_of_file_locations['text_file_location']:
    try:
        with open(path, 'r', encoding='utf-8') as file:
            all_text += file.read()
    except Exception as e:
        print(f"Error reading {path}: {e}")

In [20]:
# cleaned_text = re.sub(r'[^\u0D80-\u0DFF\s]', '', all_text)
cleaned_text = clean_words(all_text, path = False)

In [21]:
cleaned_text[826:840]

['සත්වයා',
 'පුද්ගලයා',
 'ස්ත්රිය',
 'පුරුෂයා',
 'ක්ෂස්ත්රියයා',
 'බ්රාහ්මණයා',
 'දෙවියා',
 'මාරයා',
 'යනාදි',
 'වසයෙන්',
 'පවත්නා',
 'ලොක',
 'සම්මුතිය',
 'ලොක']

In [22]:
len(cleaned_text)

229098

In [23]:
# tokens = cleaned_text.split()

In [24]:
unique_tokens = sorted(set(cleaned_text))

In [25]:
unique_tokens[:5]

['ං', 'ංකර', 'ංකාර', 'ංකෘත', 'ංඛ්යාත']

In [26]:
len(unique_tokens)

59331

## SiDiaC - v1 After Cleaning

In [27]:
def find_matching_folders(master_folder1, master_folder2, threshold=90):
    """
    Gets subfolder names from master_folder1 and fuzzy-matches them to full paths
    in master_folder2 (titles can drift slightly between snapshots, e.g. renames).
    """
    from thefuzz import fuzz, process

    # 1. Get the list of folder names in MasterFolder1
    try:
        # Get immediate subdirectories in MasterFolder1
        folder_names_1 = {
            item.name
            for item in os.scandir(master_folder1)
            if item.is_dir()
        }
    except FileNotFoundError:
        print(f"Error: Master folder 1 not found at {master_folder1}")
        return []

    if not folder_names_1:
        print(f"No folders found in {master_folder1}")
        return []

    # 2. Collect all folder names (with full paths) in MasterFolder2
    try:
        folder_paths_2 = {}
        for root, dirs, _ in os.walk(master_folder2):
            for dir_name in dirs:
                folder_paths_2.setdefault(dir_name, os.path.join(root, dir_name))
    except FileNotFoundError:
        print(f"Error: Master folder 2 not found at {master_folder2}")
        return []

    # 3. Fuzzy-match each MasterFolder1 name against MasterFolder2 folder names
    choices = list(folder_paths_2.keys())
    matching_locations = []
    overlap = []
    for name in folder_names_1:
        best_match = process.extractOne(
            name, choices, scorer=fuzz.token_sort_ratio, score_cutoff=threshold
        )
        if best_match:
            matched_name = best_match[0]
            matching_locations.append(folder_paths_2[matched_name])
            overlap.append(matched_name)

    return folder_names_1, matching_locations, overlap

In [28]:
MasterFolder1_Path = "data/OCR_Final_V1_AC" # REPLACE with your actual path
MasterFolder2_Path = "data/OCR_Final" # REPLACE with your actual path

folder_names_1, list_of_locations, overlaps = find_matching_folders(MasterFolder1_Path, MasterFolder2_Path)


In [29]:
len(list_of_locations)

40

In [30]:
absolute_locations = [
    os.path.abspath(loc) 
    for loc in list_of_locations
]

In [31]:
absolute_locations

['/Users/nevidujayatilleke/Documents/MSC - Research/OCR of Corpus/SiDiaC-v.2.0/data/OCR_Final/සාරාර්ථ සංග්\u200dරහව ප්\u200dරථම භාගය',
 '/Users/nevidujayatilleke/Documents/MSC - Research/OCR of Corpus/SiDiaC-v.2.0/data/OCR_Final/දළදා පූජාවලිය',
 '/Users/nevidujayatilleke/Documents/MSC - Research/OCR of Corpus/SiDiaC-v.2.0/data/OCR_Final/පන්සිය පනස් ජාතක පොත',
 '/Users/nevidujayatilleke/Documents/MSC - Research/OCR of Corpus/SiDiaC-v.2.0/data/OCR_Final/අධිමාස සංග්\u200dරහව',
 '/Users/nevidujayatilleke/Documents/MSC - Research/OCR of Corpus/SiDiaC-v.2.0/data/OCR_Final/දහම් සරණ.txt',
 '/Users/nevidujayatilleke/Documents/MSC - Research/OCR of Corpus/SiDiaC-v.2.0/data/OCR_Final/දුර්වාදී හෘදය විදාරණය',
 '/Users/nevidujayatilleke/Documents/MSC - Research/OCR of Corpus/SiDiaC-v.2.0/data/OCR_Final/ජුබිලි වර්ණනාව',
 '/Users/nevidujayatilleke/Documents/MSC - Research/OCR of Corpus/SiDiaC-v.2.0/data/OCR_Final/මුවදෙව් දා විවරණය',
 '/Users/nevidujayatilleke/Documents/MSC - Research/OCR of Corpus/SiD

In [32]:
def find_single_txt_file_location(start_directory):
    # Check if the starting directory is valid
    if not os.path.isdir(start_directory):
        return f"Error: Directory not found at {start_directory}"

    # Walk through the directory and its subdirectories
    for root, _, files in os.walk(start_directory):
        for file in files:
            # Check for the .txt extension (case-insensitive)
            if file.lower().endswith('.txt'):
                full_path = os.path.join(root, file)
                return full_path  # Return the first one found (which should be the only one)
                
    return "Error: No .txt file was found in the specified location."

In [33]:
sidiac = []
for loc in list_of_locations:
    file_location = find_single_txt_file_location(loc)
    temp = clean_words(file_location)
    sidiac.append(temp)

In [34]:
sidiac_flat = [item for sublist in sidiac for item in sublist]

In [35]:
len(sidiac_flat)

43959

In [36]:
abs_locations_upd = []
for loc in absolute_locations:
    temp = find_single_txt_file_location(loc)
    abs_locations_upd.append(temp)

In [37]:
# root_directory_to_scan = "data/OCR_Final_V1_AC"

# list_of_file_locations_v1 = get_written_dates_and_any_text_file_locations_to_dataframe(root_directory_to_scan)

In [38]:
list_of_file_locations['text_file_location'][0]

'/Users/nevidujayatilleke/Documents/MSC - Research/OCR of Corpus/SiDiaC-v.2.0/data/OCR_Final/ධර්ම ප්\u200dරදීපිකාව හෙවත් මහාබෝධිවංශ පරිකථාව/ධර්ම ප්\u200dරදීපිකාව හෙවත් මහාබෝධිවංශ පරිකථාව.txt'

In [39]:
filter_mask = list_of_file_locations['text_file_location'].isin(abs_locations_upd)
list_of_file_locations_v1 = list_of_file_locations[filter_mask]

In [40]:
list_of_file_locations_v1.head()

,title,written_date,text_file_location,word_count_bf,sent_count_bf
0,ධර්ම ප්‍රදීපිකාව හෙවත් මහාබෝධිවංශ පරිකථාව,1187 - 1225,/Users/nevidujayatilleke/Documents/MSC - Resea...,820,41
3,නිදහසේ මන්ත්‍රය,1901 - 1938,/Users/nevidujayatilleke/Documents/MSC - Resea...,1076,182
4,පැරණි ගම,1944,/Users/nevidujayatilleke/Documents/MSC - Resea...,1627,92
5,පන්සිය පනස් ජාතක පොත,1303 - 1333,/Users/nevidujayatilleke/Documents/MSC - Resea...,1553,53
6,හිතෝපදේශ සන්නය,1825 - 1905,/Users/nevidujayatilleke/Documents/MSC - Resea...,928,34


In [41]:
list_of_file_locations_v1['word_count_bf'].sum()

np.int64(43959)

In [42]:
# list_of_file_locations_v1['sent_count_bf'] = list_of_file_locations_v1['text_file_location'].apply(count_eos_tokens)

In [43]:
list_of_file_locations_v1['sent_count_bf'].head()

0     41
3    182
4     92
5     53
6     34
Name: sent_count_bf, dtype: int64

In [44]:
list_of_file_locations_v1['sent_count_bf'].sum()

np.int64(2978)

## Stopword Analysis

In [45]:
# from nltk.probability import FreqDist

# fdist = FreqDist(cleaned_text)

In [46]:
# fdist

In [47]:
# frequencies = list(fdist.values()) 

In [48]:
# mean_freq = np.mean(frequencies)

In [49]:
# std_dev = np.std(frequencies) 

In [50]:
# # Prepare a list to store the results
# z_score_results = []

# for word, frequency in fdist.items():
#     if std_dev == 0:
#         # Avoid division by zero if all frequencies are the same
#         z_score = 0.0
#     else:
#         # Apply the Z-score formula
#         z_score = (frequency - mean_freq) / std_dev
    
#     z_score_results.append({
#         'word': word,
#         # 'frequency': frequency,
#         'z_score': z_score
#     })

In [51]:
# sorted_results = sorted(z_score_results, key=lambda x: x['z_score'], reverse=True)

In [52]:
# sorted_results[:50]

In [53]:
# import json

In [54]:
# filename = 'z_score_analysis.json'

In [55]:
# with open(filename, 'w', encoding='utf-8') as f:
#     # The 'indent=4' argument formats the JSON output for readability
#     json.dump(sorted_results, f, indent=4, ensure_ascii=False)

## Century-wise Analaysis

In [56]:
list_of_file_locations_sdc_fil = list_of_file_locations.copy()

In [57]:
list_of_file_locations_sdc_fil.dropna(subset=['written_date'], inplace=True)

In [58]:
list_of_file_locations_sdc_fil.head(10)

,title,written_date,text_file_location,word_count_bf,sent_count_bf
0,ධර්ම ප්‍රදීපිකාව හෙවත් මහාබෝධිවංශ පරිකථාව,1187 - 1225,/Users/nevidujayatilleke/Documents/MSC - Resea...,820,41
2,ටිබැට් රටේ බුදුදහම,1832 - 1907,/Users/nevidujayatilleke/Documents/MSC - Resea...,1052,53
3,නිදහසේ මන්ත්‍රය,1901 - 1938,/Users/nevidujayatilleke/Documents/MSC - Resea...,1076,182
4,පැරණි ගම,1944,/Users/nevidujayatilleke/Documents/MSC - Resea...,1627,92
5,පන්සිය පනස් ජාතක පොත,1303 - 1333,/Users/nevidujayatilleke/Documents/MSC - Resea...,1553,53
6,හිතෝපදේශ සන්නය,1825 - 1905,/Users/nevidujayatilleke/Documents/MSC - Resea...,928,34
9,අධිමාස සංග්‍රහව,1850 - 1903,/Users/nevidujayatilleke/Documents/MSC - Resea...,606,22
11,දහම් සරණ.txt,1220 - 1293,/Users/nevidujayatilleke/Documents/MSC - Resea...,1901,55
15,දුර්වාදී හෘදය විදාරණය,1854 - 1899,/Users/nevidujayatilleke/Documents/MSC - Resea...,1855,125
18,විස්තරාර්ථ ග්‍රන්ථිපද විවර්ණය සහිත සිංහල විමාන...,1469 - 1815,/Users/nevidujayatilleke/Documents/MSC - Resea...,1085,26


In [59]:
len(list_of_file_locations_sdc_fil)

59

In [60]:
list_of_file_locations_sdc_fil.reset_index(drop=True, inplace=True)

In [61]:
list_of_file_locations_sdc_fil.head()

,title,written_date,text_file_location,word_count_bf,sent_count_bf
0,ධර්ම ප්‍රදීපිකාව හෙවත් මහාබෝධිවංශ පරිකථාව,1187 - 1225,/Users/nevidujayatilleke/Documents/MSC - Resea...,820,41
1,ටිබැට් රටේ බුදුදහම,1832 - 1907,/Users/nevidujayatilleke/Documents/MSC - Resea...,1052,53
2,නිදහසේ මන්ත්‍රය,1901 - 1938,/Users/nevidujayatilleke/Documents/MSC - Resea...,1076,182
3,පැරණි ගම,1944,/Users/nevidujayatilleke/Documents/MSC - Resea...,1627,92
4,පන්සිය පනස් ජාතක පොත,1303 - 1333,/Users/nevidujayatilleke/Documents/MSC - Resea...,1553,53


In [62]:
list_of_file_locations_sdc_fil['word_count_bf'].sum()

np.int64(64805)

In [63]:
list_of_file_locations_sdc_fil['sent_count_bf'].sum()

np.int64(4371)

In [64]:
all_text_sdc_fil = ""
for path in list_of_file_locations_sdc_fil['text_file_location']:
    try:
        with open(path, 'r', encoding='utf-8') as file:
            all_text_sdc_fil += file.read()
    except Exception as e:
        print(f"Error reading {path}: {e}")

In [65]:
cleaned_text_sdc_fil = clean_words(all_text_sdc_fil, path = False)

In [66]:
dataframe_corpora_comp = pd.DataFrame({
    'SDC': {
        'Total Files': len(list_of_file_locations), 
        'Total Words': list_of_file_locations['word_count_bf'].sum(),
        'Unique Words': len(set(cleaned_text)),
        'Total Sentences': list_of_file_locations['sent_count_bf'].sum(),
    },
    'SDC-Filtered': {
        'Total Files': len(list_of_file_locations_sdc_fil), 
        'Total Words': list_of_file_locations_sdc_fil['word_count_bf'].sum(),
        'Unique Words': len(set(cleaned_text_sdc_fil)),
        'Total Sentences': list_of_file_locations_sdc_fil['sent_count_bf'].sum()
    },
    'SiDiaC': {
        'Total Files': 46, 
        'Total Words': 58027,
        'Unique Words': 22837,
        'Total Sentences': str('-'),
    },
    'SiDiaC-Filtered': {
        'Total Files': len(list_of_file_locations_v1),
        'Total Words': list_of_file_locations_v1['word_count_bf'].sum(),
        'Unique Words': len(set(sidiac_flat)),
        'Total Sentences': list_of_file_locations_v1['sent_count_bf'].sum(),
    }
})

In [67]:
dataframe_corpora_comp

,SDC,SDC-Filtered,SiDiaC,SiDiaC-Filtered
Total Files,185,59,46,40
Total Words,229098,64805,58027,43959
Unique Words,59331,22031,22837,16209
Total Sentences,13523,4371,-,2978


In [68]:
dataframe_corpora_comp.to_latex('corpora_comparison.tex', index=True, header=True, encoding='utf-8')

In [69]:
dataframe_corpora_comp = pd.DataFrame({
    'Total Files': {
        'SDC': len(list_of_file_locations), 
        'SDC-Filtered': len(list_of_file_locations_sdc_fil),
        'SiDiaC': 46,
        'SiDiaC-Filtered': len(list_of_file_locations_v1),
    },
    'Total Words': {
        'SDC': list_of_file_locations['word_count_bf'].sum(), 
        'SDC-Filtered': list_of_file_locations_sdc_fil['word_count_bf'].sum(),
        'SiDiaC': 58027,
        'SiDiaC-Filtered': list_of_file_locations_v1['word_count_bf'].sum()
    },
    'Unique Words': {
        'SDC': len(set(cleaned_text)), 
        'SDC-Filtered': len(set(cleaned_text_sdc_fil)),
        'SiDiaC': 22837,
        'SiDiaC-Filtered': len(set(sidiac_flat)),
    },
    'Total Sentences': {
        'SDC': list_of_file_locations['sent_count_bf'].sum(),
        'SDC-Filtered': list_of_file_locations_sdc_fil['sent_count_bf'].sum(),
        'SiDiaC': str('-'),
        'SiDiaC-Filtered': list_of_file_locations_v1['sent_count_bf'].sum(),
    }
})

In [70]:
dataframe_corpora_comp

,Total Files,Total Words,Unique Words,Total Sentences
SDC,185,229098,59331,13523
SDC-Filtered,59,64805,22031,4371
SiDiaC,46,58027,22837,-
SiDiaC-Filtered,40,43959,16209,2978


In [71]:
dataframe_corpora_comp.to_latex('corpora_comparison.tex', index=True, header=True, encoding='utf-8')

In [72]:
list_of_file_locations_sdc_fil.sort_values(by='title').reset_index(drop=True).to_clipboard()

In [73]:
list_of_file_locations['start_date'] = np.nan
list_of_file_locations['end_date'] = np.nan

In [74]:
# list_of_file_locations[list_of_file_locations['written_date'] == '1341 – 1408']['text_file_location'].iloc[0]

In [75]:
# ans

In [76]:
for index, row in list_of_file_locations_sdc_fil.iterrows():
    date_str = str(row['written_date']) # Ensure it's treated as a string
    
    start_year = np.nan
    end_year = np.nan

    if ' - ' in date_str:
        # It's a range
        try:
            start_year_str, end_year_str = date_str.split(' - ')
            start_year = int(start_year_str.strip())
            end_year = int(end_year_str.strip())
        except ValueError:
            print(f"Warning: Could not parse range '{date_str}' at index {index}. Setting to NaN.")
    else:
        # It's a single year or potentially unparseable
        try:
            single_year = int(date_str.strip())
            start_year = single_year
            end_year = single_year # Duplicate for single date
        except ValueError:
            print(f"Warning: Could not parse single year/date '{date_str}' at index {index}. Setting to NaN.")
            
    # Assign the parsed (or NaN) values back to the DataFrame
    list_of_file_locations_sdc_fil.at[index, 'start_date'] = start_year
    list_of_file_locations_sdc_fil.at[index, 'end_date'] = end_year


In [77]:
list_of_file_locations_sdc_fil.head()

,title,written_date,text_file_location,word_count_bf,sent_count_bf,start_date,end_date
0,ධර්ම ප්‍රදීපිකාව හෙවත් මහාබෝධිවංශ පරිකථාව,1187 - 1225,/Users/nevidujayatilleke/Documents/MSC - Resea...,820,41,1187.0,1225.0
1,ටිබැට් රටේ බුදුදහම,1832 - 1907,/Users/nevidujayatilleke/Documents/MSC - Resea...,1052,53,1832.0,1907.0
2,නිදහසේ මන්ත්‍රය,1901 - 1938,/Users/nevidujayatilleke/Documents/MSC - Resea...,1076,182,1901.0,1938.0
3,පැරණි ගම,1944,/Users/nevidujayatilleke/Documents/MSC - Resea...,1627,92,1944.0,1944.0
4,පන්සිය පනස් ජාතක පොත,1303 - 1333,/Users/nevidujayatilleke/Documents/MSC - Resea...,1553,53,1303.0,1333.0


In [78]:
def get_century(year):
    """Calculates the century for a given year."""
    if pd.isna(year):
        return np.nan
    return (int(year) - 1) // 100 + 1

In [79]:
list_of_file_locations_sdc_fil['end_century'] = list_of_file_locations_sdc_fil['end_date'].apply(get_century)

In [80]:
list_of_file_locations_sdc_fil.head()

,title,written_date,text_file_location,word_count_bf,sent_count_bf,start_date,end_date,end_century
0,ධර්ම ප්‍රදීපිකාව හෙවත් මහාබෝධිවංශ පරිකථාව,1187 - 1225,/Users/nevidujayatilleke/Documents/MSC - Resea...,820,41,1187.0,1225.0,13
1,ටිබැට් රටේ බුදුදහම,1832 - 1907,/Users/nevidujayatilleke/Documents/MSC - Resea...,1052,53,1832.0,1907.0,20
2,නිදහසේ මන්ත්‍රය,1901 - 1938,/Users/nevidujayatilleke/Documents/MSC - Resea...,1076,182,1901.0,1938.0,20
3,පැරණි ගම,1944,/Users/nevidujayatilleke/Documents/MSC - Resea...,1627,92,1944.0,1944.0,20
4,පන්සිය පනස් ජාතක පොත,1303 - 1333,/Users/nevidujayatilleke/Documents/MSC - Resea...,1553,53,1303.0,1333.0,14


In [81]:
# def count_clean_words(path):
#     try:
#         with open(path, 'r', encoding='utf-8') as file:
#             text = file.read()

#         # # Keep Sinhala + Latin characters + whitespace, remove others
#         # cleaned = re.sub(r'[^\u0D80-\u0DFFa-zA-Z\s]', '', text)
#         # Keep only Sinhala characters and whitespace, remove Latin and others
#         cleaned = re.sub(r'[^\u0D80-\u0DFF\s]', '', text)

#         # Tokenize by whitespace
#         words = cleaned.split()
#         # print(words)
#         return len(words)

#     except Exception as e:
#         print(f"Error reading {path}: {e}")
#         return 0

In [82]:
# list_of_file_locations_sdc_fil['word_count_af'] = list_of_file_locations_sdc_fil['text_file_location'].apply(lambda x: len(clean_words(x)))

In [83]:
# list_of_file_locations_sdc_fil['word_count_af'].head()

In [84]:
list_of_file_locations_sdc_fil['word_count_bf'].head()

0     820
1    1052
2    1076
3    1627
4    1553
Name: word_count_bf, dtype: int64

In [85]:
century_tokens = list_of_file_locations_sdc_fil.groupby('end_century')['word_count_bf'].sum().reset_index()

In [86]:
century_tokens

,end_century,word_count_bf
0,5,565
1,12,866
2,13,13614
3,14,5210
4,15,8483
5,16,1369
6,18,4707
7,19,12698
8,20,17293


In [87]:
century_tokens['word_count_bf'].sum()

np.int64(64805)

In [88]:
list_of_file_locations_sdc_fil

,title,written_date,text_file_location,word_count_bf,sent_count_bf,start_date,end_date,end_century
0,ධර්ම ප්‍රදීපිකාව හෙවත් මහාබෝධිවංශ පරිකථාව,1187 - 1225,/Users/nevidujayatilleke/Documents/MSC - Resea...,820,41,1187.0,1225.0,13
1,ටිබැට් රටේ බුදුදහම,1832 - 1907,/Users/nevidujayatilleke/Documents/MSC - Resea...,1052,53,1832.0,1907.0,20
2,නිදහසේ මන්ත්‍රය,1901 - 1938,/Users/nevidujayatilleke/Documents/MSC - Resea...,1076,182,1901.0,1938.0,20
3,පැරණි ගම,1944,/Users/nevidujayatilleke/Documents/MSC - Resea...,1627,92,1944.0,1944.0,20
4,පන්සිය පනස් ජාතක පොත,1303 - 1333,/Users/nevidujayatilleke/Documents/MSC - Resea...,1553,53,1303.0,1333.0,14
5,හිතෝපදේශ සන්නය,1825 - 1905,/Users/nevidujayatilleke/Documents/MSC - Resea...,928,34,1825.0,1905.0,20
6,අධිමාස සංග්‍රහව,1850 - 1903,/Users/nevidujayatilleke/Documents/MSC - Resea...,606,22,1850.0,1903.0,20
7,දහම් සරණ.txt,1220 - 1293,/Users/nevidujayatilleke/Documents/MSC - Resea...,1901,55,1220.0,1293.0,13
8,දුර්වාදී හෘදය විදාරණය,1854 - 1899,/Users/nevidujayatilleke/Documents/MSC - Resea...,1855,125,1854.0,1899.0,19
9,විස්තරාර්ථ ග්‍රන්ථිපද විවර්ණය සහිත සිංහල විමාන...,1469 - 1815,/Users/nevidujayatilleke/Documents/MSC - Resea...,1085,26,1469.0,1815.0,19


In [89]:
page_counts = pd.read_csv('page_counts.csv')

In [90]:
page_counts.head()

,title,pages
0,අංගුත්තරනිකායෙහි පළමුවෙනි කාණ්ඩය එකක සහ දුක නි...,5
1,අජූද නීතිය,5
2,අටුවා සන්න සහිත ධම්මික සූත්‍රය,11
3,අධිකරණ විභාග සංඝෝ,5
4,අධිමාස දීපනය,7


In [91]:
page_counts['title'].iloc[0]

'අංගුත්තරනිකායෙහි පළමුවෙනි කාණ්ඩය එකක සහ දුක නිපාතයෝ'

In [92]:
from thefuzz import fuzz, process

In [93]:
def fuzzy_left_join(df_left, df_right, key_col, threshold=90):
    # 1. Prepare the list of titles to search against from the right DataFrame
    choices = df_right[key_col].tolist()
    
    # 2. Iterate through the left DataFrame to find the best fuzzy match
    match_data = []
    
    for index, row in df_left.iterrows():
        title_to_match = row[key_col]
        
        # Use token_sort_ratio for flexible matching
        best_match = process.extractOne(
            title_to_match, 
            choices, 
            scorer=fuzz.token_sort_ratio, 
            score_cutoff=threshold # Only considers scores >= threshold
        )
        
        # Record the key from the left DF, and if a match was found, record the match details.
        # If no match is found, the match fields will be None, leading to NaN after merge.
        match_info = {
            f'{key_col}_left': title_to_match,
            'original_index': index
        }
        
        if best_match:
            match_info.update({
                f'{key_col}_match': best_match[0],
                'match_score': best_match[1]
            })
        
        match_data.append(match_info)

    # 3. Create a temporary DataFrame of matches
    df_matches = pd.DataFrame(match_data)
    
    # 4. Perform the final pandas LEFT merge (Join the left DF to the matches)
    #    We use 'left' to ensure ALL rows from df_left are kept.
    df_merged = df_left.reset_index().merge(
        df_matches, 
        left_on='index', 
        right_on='original_index', 
        how='left' # <-- Key difference: keep all left rows!
    ).set_index('index')
    
    # 5. Join the result to the right DF based on the matched title
    #    This is also a LEFT merge to keep all records from the current df_merged.
    final_df = pd.merge(
        df_merged,
        df_right.rename(columns={key_col: f'{key_col}_right'}),
        left_on=f'{key_col}_match',
        right_on=f'{key_col}_right',
        how='left', # <-- Keep all left rows, even if they didn't have a fuzzy match
        suffixes=('_L', '_R')
    )
    
    # # Clean up the temporary columns
    # final_df = final_df.drop(columns=['original_index', f'{key_col}_match', f'{key_col}_right'])
    
    # # Rename the left's key_col back to the original name
    # final_df = final_df.rename(columns={f'{key_col}_left': f'{key_col}'})

    columns_to_drop = [
            'original_index', 
            f'{key_col}_match', 
            f'{key_col}_right', 
            f'{key_col}_left',
            'match_score'
        ]
    
    # Remove the columns
    final_df = final_df.drop(columns=columns_to_drop)
    
    return final_df.reset_index(drop=True)

In [94]:
result_df = fuzzy_left_join(list_of_file_locations_sdc_fil, page_counts, key_col='title', threshold=90)

In [95]:
result_df

,title,written_date,text_file_location,word_count_bf,sent_count_bf,start_date,end_date,end_century,pages
0,ධර්ම ප්‍රදීපිකාව හෙවත් මහාබෝධිවංශ පරිකථාව,1187 - 1225,/Users/nevidujayatilleke/Documents/MSC - Resea...,820,41,1187.0,1225.0,13,5.0
1,ටිබැට් රටේ බුදුදහම,1832 - 1907,/Users/nevidujayatilleke/Documents/MSC - Resea...,1052,53,1832.0,1907.0,20,7.0
2,නිදහසේ මන්ත්‍රය,1901 - 1938,/Users/nevidujayatilleke/Documents/MSC - Resea...,1076,182,1901.0,1938.0,20,8.0
3,පැරණි ගම,1944,/Users/nevidujayatilleke/Documents/MSC - Resea...,1627,92,1944.0,1944.0,20,5.0
4,පන්සිය පනස් ජාතක පොත,1303 - 1333,/Users/nevidujayatilleke/Documents/MSC - Resea...,1553,53,1303.0,1333.0,14,5.0
5,හිතෝපදේශ සන්නය,1825 - 1905,/Users/nevidujayatilleke/Documents/MSC - Resea...,928,34,1825.0,1905.0,20,6.0
6,අධිමාස සංග්‍රහව,1850 - 1903,/Users/nevidujayatilleke/Documents/MSC - Resea...,606,22,1850.0,1903.0,20,5.0
7,දහම් සරණ.txt,1220 - 1293,/Users/nevidujayatilleke/Documents/MSC - Resea...,1901,55,1220.0,1293.0,13,NaN
8,දුර්වාදී හෘදය විදාරණය,1854 - 1899,/Users/nevidujayatilleke/Documents/MSC - Resea...,1855,125,1854.0,1899.0,19,5.0
9,විස්තරාර්ථ ග්‍රන්ථිපද විවර්ණය සහිත සිංහල විමාන...,1469 - 1815,/Users/nevidujayatilleke/Documents/MSC - Resea...,1085,26,1469.0,1815.0,19,5.0


In [96]:
result_df['title'][result_df['pages'].isna()]

7                    දහම් සරණ.txt
16           ධම්පියා අටුවා ගැටපදය
17    සන්න සහිත සැලළිහිණි සන්දේශය
28                   කාව්‍යෙශ්ඛරය
Name: title, dtype: object

In [97]:
replace_page_counts = {'ධම්පියා අටුවා ගැටපදය': 5,
                       'සන්න සහිත සැලළිහිණි සන්දේශය': 5,
                       'කාව්‍යෙශ්ඛරය':5,
                       'ධර්මප්‍රදීපිකාව': 5}

In [98]:
title_mask = result_df['title'].isin(replace_page_counts.keys())
result_df.loc[title_mask, 'pages'] = result_df.loc[title_mask, 'title'].map(replace_page_counts)

In [99]:
result_df['title'][result_df['pages'].isna()]

7                    දහම් සරණ.txt
17    සන්න සහිත සැලළිහිණි සන්දේශය
Name: title, dtype: object

In [100]:
result_df['pages'].value_counts()

pages
5.0     34
6.0     13
7.0      8
8.0      1
15.0     1
Name: count, dtype: int64

In [101]:
result_df['avg_word_count_bf'] = result_df['word_count_bf'] / result_df['pages']

In [102]:
result_df['avg_sent_count_bf'] = result_df['sent_count_bf'] / result_df['pages']

In [103]:
result_df['pages_needed'] = np.ceil((4000 / result_df['avg_word_count_bf']) - result_df['pages'])

In [104]:
result_df.head()

,title,written_date,text_file_location,word_count_bf,sent_count_bf,start_date,end_date,end_century,pages,avg_word_count_bf,avg_sent_count_bf,pages_needed
0,ධර්ම ප්‍රදීපිකාව හෙවත් මහාබෝධිවංශ පරිකථාව,1187 - 1225,/Users/nevidujayatilleke/Documents/MSC - Resea...,820,41,1187.0,1225.0,13,5.0,164.000000,8.200000,20.0
1,ටිබැට් රටේ බුදුදහම,1832 - 1907,/Users/nevidujayatilleke/Documents/MSC - Resea...,1052,53,1832.0,1907.0,20,7.0,150.285714,7.571429,20.0
2,නිදහසේ මන්ත්‍රය,1901 - 1938,/Users/nevidujayatilleke/Documents/MSC - Resea...,1076,182,1901.0,1938.0,20,8.0,134.500000,22.750000,22.0
3,පැරණි ගම,1944,/Users/nevidujayatilleke/Documents/MSC - Resea...,1627,92,1944.0,1944.0,20,5.0,325.400000,18.400000,8.0
4,පන්සිය පනස් ජාතක පොත,1303 - 1333,/Users/nevidujayatilleke/Documents/MSC - Resea...,1553,53,1303.0,1333.0,14,5.0,310.600000,10.600000,8.0


In [105]:
result_df

,title,written_date,text_file_location,word_count_bf,sent_count_bf,start_date,end_date,end_century,pages,avg_word_count_bf,avg_sent_count_bf,pages_needed
0,ධර්ම ප්‍රදීපිකාව හෙවත් මහාබෝධිවංශ පරිකථාව,1187 - 1225,/Users/nevidujayatilleke/Documents/MSC - Resea...,820,41,1187.0,1225.0,13,5.0,164.000000,8.200000,20.0
1,ටිබැට් රටේ බුදුදහම,1832 - 1907,/Users/nevidujayatilleke/Documents/MSC - Resea...,1052,53,1832.0,1907.0,20,7.0,150.285714,7.571429,20.0
2,නිදහසේ මන්ත්‍රය,1901 - 1938,/Users/nevidujayatilleke/Documents/MSC - Resea...,1076,182,1901.0,1938.0,20,8.0,134.500000,22.750000,22.0
3,පැරණි ගම,1944,/Users/nevidujayatilleke/Documents/MSC - Resea...,1627,92,1944.0,1944.0,20,5.0,325.400000,18.400000,8.0
4,පන්සිය පනස් ජාතක පොත,1303 - 1333,/Users/nevidujayatilleke/Documents/MSC - Resea...,1553,53,1303.0,1333.0,14,5.0,310.600000,10.600000,8.0
5,හිතෝපදේශ සන්නය,1825 - 1905,/Users/nevidujayatilleke/Documents/MSC - Resea...,928,34,1825.0,1905.0,20,6.0,154.666667,5.666667,20.0
6,අධිමාස සංග්‍රහව,1850 - 1903,/Users/nevidujayatilleke/Documents/MSC - Resea...,606,22,1850.0,1903.0,20,5.0,121.200000,4.400000,29.0
7,දහම් සරණ.txt,1220 - 1293,/Users/nevidujayatilleke/Documents/MSC - Resea...,1901,55,1220.0,1293.0,13,NaN,NaN,NaN,NaN
8,දුර්වාදී හෘදය විදාරණය,1854 - 1899,/Users/nevidujayatilleke/Documents/MSC - Resea...,1855,125,1854.0,1899.0,19,5.0,371.000000,25.000000,6.0
9,විස්තරාර්ථ ග්‍රන්ථිපද විවර්ණය සහිත සිංහල විමාන...,1469 - 1815,/Users/nevidujayatilleke/Documents/MSC - Resea...,1085,26,1469.0,1815.0,19,5.0,217.000000,5.200000,14.0


In [106]:
result_df[['title', 'written_date', 'word_count_bf', 'sent_count_bf', 'pages', 'avg_word_count_bf', 'avg_sent_count_bf', 'pages_needed']].to_csv('corpus_analysis_with_page_counts.csv', index=False)

In [107]:
poetry_books = ['නිදහසේ මන්ත්‍රය',
                'ජුබිලි වර්ණනාව',
                'හංස සංදේශය',
                'කාව්‍ය වජ්‍රායුධය - පළමු කොටස',
                'ලංකා මාතා',
                'මහියංගන වර්ණනාව',
                'සිතියම් සහිත සියබස් මල්දම',
                'කාව්‍යෙශ්ඛරය',
                'යොග රත්නාකරය',
                'කුදුසික',
                'පරවි සංදේශය',
                'මගේ මල්ලි',
                'සන්න සහිත කව්සිළුමිණ හෙවත් කුසදාවත',
                'ලියනෝරා නාඩගම',
                'නිකම් හැකියාව']

In [108]:
poetry_books = result_df[result_df['title'].isin(poetry_books)][['title', 'written_date', 'word_count_bf', 'sent_count_bf', 'pages', 'avg_word_count_bf', 'avg_sent_count_bf', 'pages_needed']]

In [109]:
poetry_books.head()

,title,written_date,word_count_bf,sent_count_bf,pages,avg_word_count_bf,avg_sent_count_bf,pages_needed
12,ජුබිලි වර්ණනාව,1857 - 1922,345,34,5.0,69.000000,6.800000,53.0
20,ලංකා මාතා,1901 - 1951,761,86,7.0,108.714286,12.285714,30.0
22,මහියංගන වර්ණනාව,1878,604,9,5.0,120.800000,1.800000,29.0
26,සිතියම් සහිත සියබස් මල්දම,1820,978,39,6.0,163.000000,6.500000,19.0
28,කාව්‍යෙශ්ඛරය,1408 - 1491,1294,139,5.0,258.800000,27.800000,11.0


In [110]:
poetry_books['word_count_bf'].sum()

np.int64(6072)

In [111]:
# def extract_sinhala_tokens(path):
#     try:
#         with open(path, 'r', encoding='utf-8') as f:
#             text = f.read()
#         # Keep Sinhala characters and spaces
#         text = re.sub(r'[^\u0D80-\u0DFF\s]', '', text)
#         return text.split()
#     except Exception as e:
#         print(f"Error reading {path}: {e}")
#         return []

In [112]:
# list_of_file_locations['tokens'] = list_of_file_locations['text_file_location'].apply(extract_sinhala_tokens)

In [113]:
# list_of_file_locations['tokens'][0]

In [114]:
# list_of_file_locations

In [115]:
# def load_stopwords(file_path):
#     try:
#         with open(file_path, 'r', encoding='utf-8') as f:
#             stopwords = f.read().splitlines()
#             # Remove empty lines and strip whitespace
#             stopwords = [word.strip() for word in stopwords if word.strip()]
#             return set(stopwords)  # set is faster for lookups
#     except Exception as e:
#         print(f"Error loading stopwords from {file_path}: {e}")
#         return set()

In [116]:
# sinhala_stopwords = load_stopwords("stop words.txt")

In [117]:
# sinhala_stopwords

In [118]:
# list_of_file_locations['filtered_tokens'] = list_of_file_locations['tokens'].apply(
#     lambda tokens: [word for word in tokens if word not in sinhala_stopwords]
# )

In [119]:
# all_tokens = sum(list_of_file_locations['filtered_tokens'], [])

In [120]:
# from collections import Counter
# from wordcloud import WordCloud
# import matplotlib.pyplot as plt

In [121]:
# counter = Counter(all_tokens)
# top_words = counter.most_common(100)

In [122]:
# top_words

In [123]:
# words, freqs = zip(*top_words)
# plt.figure(figsize=(12, 6))
# plt.bar(words, freqs, color='mediumseagreen')
# plt.xticks(rotation=45, ha='right', fontsize=13)
# plt.ylabel("Frequency")
# plt.title("Top 20 Sinhala Words")
# plt.tight_layout()
# plt.show()

In [124]:
# wordcloud = WordCloud(
#     font_path="NotoSerifSinhala-VariableFont_wdth.ttf",  # Make sure Sinhala font is available
#     background_color='white',
#     width=800,
#     height=400
# ).generate_from_frequencies(counter)

# plt.figure(figsize=(12, 6))
# plt.imshow(wordcloud, interpolation='bilinear')
# plt.axis('off')
# plt.title("Sinhala Word Cloud", fontsize=16)
# plt.show()

In [125]:
# century_tokens = list_of_file_locations.groupby('end_century')['tokens'].sum()  # flatten lists
# century_tokens = century_tokens.to_dict()

In [126]:
# century_tokens

In [127]:
# len(century_tokens[5])

In [128]:
# len(list(set(century_tokens[5])))

In [129]:
# len(century_tokens[13])

In [130]:
# len(list(set(century_tokens[13])))

In [131]:
# len(century_tokens[14])

In [132]:
# len(list(set(century_tokens[14])))

In [133]:
# len(century_tokens[15])

In [134]:
# len(list(set(century_tokens[15])))

In [135]:
# len(century_tokens[18])

In [136]:
# len(list(set(century_tokens[18])))

In [137]:
# len(century_tokens[19])

In [138]:
# len(list(set(century_tokens[19])))

In [139]:
# len(century_tokens[20])

In [140]:
# len(list(set(century_tokens[20])))

In [141]:
# # Get total frequency of each word per century
# freq_by_century = {
#     century: Counter(tokens)
#     for century, tokens in century_tokens.items()
# }

# # Create a DataFrame: rows = words, columns = centuries
# freq_df = pd.DataFrame(freq_by_century).fillna(0).astype(int)

In [142]:
# freq_df

In [143]:
# freq_df.loc['නිදහසේ']

In [144]:
# freq_df[20].describe()

In [145]:
# zscore_df = freq_df.copy()
# zscore_df = (freq_df - freq_df.mean()) / freq_df.std(ddof=0)

In [146]:
# zscore_df.loc['නිදහසේ']

In [147]:
# zscore_df.head()

In [148]:
# zscore_df.to_csv("Z_Scores.csv")

In [149]:
# def compute_coverage(zscores, lower=-1.5, upper=1.5):
#     total = zscores.size
#     within = ((zscores > lower) & (zscores < upper)).sum().sum()
#     return within / total

In [150]:
# coverage = compute_coverage(zscore_df, lower=float('-inf'), upper=6.1027)

In [151]:
# print(f"Coverage for z ∈ [-1.5, 1.5]: {coverage*100:.2f}%")

In [152]:
# typical_words = {
#     century: zscore_df[
#         zscore_df[century].between(-1.5, 1.5)
#     ].index.tolist()
#     for century in zscore_df.columns
# }

In [153]:
# typical_words

In [154]:
# len(typical_words[5])

In [155]:
# distinctive_words = {
#     century: zscore_df[
#         ~zscore_df[century].between(float('-inf'), 6.1027)
#     ].sort_values(by=century, ascending=False).index.tolist()
#     for century in zscore_df.columns
# }

In [156]:
# distinctive_words

In [157]:
# distinctive_words[20]

In [158]:
# len(distinctive_words[5])

In [159]:
# len(distinctive_words[13])

In [160]:
# len(distinctive_words[14])

In [161]:
# len(distinctive_words[15])

In [162]:
# len(distinctive_words[18])

In [163]:
# len(distinctive_words[19])

In [164]:
# len(distinctive_words[20])

In [165]:
# zscore_df.head()

In [166]:
# zscore_dict_cen = zscore_df[19].to_dict()

In [167]:
# word_scores = sorted(zscore_dict_cen.items(), key = lambda item: item[1], reverse=True)

In [168]:
# with open("stopwords_19th.txt", "w", encoding="utf-8") as f:
#     for word, score in word_scores:
#         f.write(f"{word}\t{score:.4f}\n")

In [169]:
# distinctive_words[5]

In [170]:
# first_10_words_per_century = {
#     century: words[:10] for century, words in distinctive_words.items()
# }

In [171]:
# first_10_words_per_century

In [172]:
# union_words = {word for words_list in distinctive_words.values() for word in words_list}

In [173]:
# sorted_union = sorted(
#     union_words,
#     key=lambda word: zscore_df.loc[word].mean(),
#     reverse=True
# )

In [174]:
# sorted_union

In [175]:
# len(sorted_union)

In [176]:
# data_for_df = {}

# for century, words_list in distinctive_words.items():
#     availability_list = []
#     for word in sorted_union:
#         if word in words_list:
#             availability_list.append(1)  # 1 for present
#         else:
#             availability_list.append(0)  # 0 for not present
    
#     # Add the list of 1s and 0s to the dictionary
#     data_for_df[century] = availability_list

In [177]:
# availability_df = pd.DataFrame(data_for_df, index=sorted_union)

In [178]:
# availability_df

In [179]:
# availability_df.to_csv('Stopwords_Distribution.csv', index=True)